# Messi peer distance

A reproducible audit of Lionel Messi against qualified same-season La Liga forwards. The canonical calculations live in `analytics/src/peer_model.py`; this notebook exposes them for inspection without duplicating the method.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / 'analytics').exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / 'analytics' / 'src'))

from peer_model import (
    METRICS, add_per90, build_peer_distance, build_sensitivity,
    data_quality, load_raw, qualified_forwards,
)

## Load and validate
The primary population is forwards with at least 900 minutes. Missingness, duplicate IDs and Messi coverage are checked before any result is interpreted.

In [ ]:
raw = load_raw()
enriched = add_per90(raw)
cohort = qualified_forwards(enriched)
quality = data_quality(raw, cohort)
pd.DataFrame(quality['by_season'])

## Primary result
Standard z is the requested distance from the arithmetic mean. Robust z and percentile are retained because football rate distributions are skewed.

In [ ]:
results = build_peer_distance(cohort)
results.sort_values('z_score', ascending=False)[
    ['season', 'metric_label', 'messi_value', 'peer_mean', 'peer_std',
     'z_score', 'robust_z', 'percentile', 'rank', 'cohort_size']
].head(20).round(3)

In [ ]:
matrix = results.pivot(index='metric_label', columns='season', values='z_score')
matrix.round(2)

## Cohort sensitivity
Every metric-season is rerun at 600, 900 and 1,350 minutes for forwards and for forwards-plus-midfielders. We report ranges rather than pretending one cutoff is uniquely correct.

In [ ]:
sensitivity = build_sensitivity(enriched)
stability = sensitivity.groupby(['season', 'metric_label']).agg(
    min_z=('z_score', 'min'), max_z=('z_score', 'max'),
    min_percentile=('percentile', 'min'), worst_rank=('rank', 'max'),
).reset_index()
stability['spread'] = stability.max_z - stability.min_z
stability.sort_values('min_z', ascending=False).head(20).round(3)

In [ ]:
pd.Series({
    'cells above +2 SD in every scenario': int((stability.min_z >= 2).sum()),
    'cells at/above 95th percentile in every scenario': int((stability.min_percentile >= 95).sum()),
    'cells top 10 in every scenario': int((stability.worst_rank <= 10).sum()),
    'total metric-seasons': len(stability),
})